In [1]:
from datetime import datetime
from pathlib import Path

from earthcode.metadata_input_definitions import ProductCollectionMetadata
from earthcode.static import create_product_collection
from earthcode.git_add import save_product_collection_to_catalog
import geopandas as gpd
import pandas as pd
from shapely.geometry import box, mapping
import pystac
from pystac.extensions.projection import ProjectionExtension
import datetime
import fsspec
import io
import numpy as np
from earthcode.static import generic_stac_item_to_earthcode_stac_item
from earthcode.git_add import save_item_to_product_collection
from earthcode.validator import validate_catalog

ImportError: cannot import name 'generic_stac_item_to_earthcode_stac_item' from 'earthcode.static' (/data/Aldhani/users/potzschf/conda/envs/openstacupload/lib/python3.12/site-packages/earthcode/static.py)

In [ ]:
catalog_root = str(Path("../open-science-catalog-metadata").resolve())

In [ ]:
product_id = "eoagritwin-crop-phenology-tomato"
product_title = "Po Valley Industrial Tomato Phenology & Crop Classification Vector Layer (Sentinel-2)"
product_description = (
    "This dataset contains the vector GeoParquet product for crop classification "
    "(Croci et al., 2026, in preparation) and phenological phase estimation "
    "(transplanting date / Start of Season and harvest date / End of Season, "
    "Ragazzi et al., 2026, in preparation) for industrial tomato "
    "(Solanum lycopersicum) over the Po Valley (Italy) for the years 2020 through 2025. "
    "Daily Sentinel-2 NDVI time series are restricted to a seasonal window "
    "(1 March - 31 October), smoothed using Radial Basis Function (RBF) interpolation, "
    "and processed via an Ensemble Machine Learning model combining derivative, MACD, "
    "min-max amplitude, and hysteresis methods calibrated on ground-truth observations "
    "in the EOAgriTwin project."
)
product_status = "ongoing"

product_license = "CC-BY-4.0"
license_link = None

product_keywords = [
    "Crop Phenology",
    "Industrial Tomato",
    "Crop Classification",
    "Sentinel-2",
    "Po Valley",
]

product_w = [7.0175]
product_s = [43.7535]
product_e = [12.748]
product_n = [45.9059]

product_start_year = 2020
product_start_month = 1
product_start_day = 1
product_end_year = 2025
product_end_month = 12
product_end_day = 31

product_region = "Po Valley, Italy"

product_themes = ["land"]

product_missions = [
    "sentinel-2",
]

product_variables = [
    "crop-type-masks",
    "land-use-land-cover-classification",

]

product_parameters = []

product_doi = None

project_id = "eoagritwin"
project_title = "EOAgriTwin - Earth Observation based Digital Twin for Resilient Agriculture under Multiple Stressors"

In [ ]:
# create a product
product_bbox = [[w, s, e, n] for s, w, n, e in zip(product_s, product_w, product_n, product_e)]

product_metadata = ProductCollectionMetadata(
    product_id=product_id,
    product_title=product_title,
    product_description=product_description,
    product_bbox=product_bbox,
    product_start_datetime=datetime(product_start_year, product_start_month, product_start_day),
    product_end_datetime=datetime(product_end_year, product_end_month, product_end_day),
    product_license=product_license,
    product_keywords=product_keywords,
    product_status=product_status,
    product_region=product_region,
    product_themes=product_themes,
    product_missions=product_missions,
    product_variables=product_variables,
    project_id=project_id,
    project_title=project_title,
    product_parameters=product_parameters,
    product_doi=product_doi,
    access_link=f'https://opensciencedata.esa.int/products/{product_id}/collection',
    license_link=license_link or None,
)

product_collection = create_product_collection(product_metadata)


In [ ]:
# save the product in local fork of the open-science-catalog-metadata repository 
catalog_root = Path(catalog_root)
save_product_collection_to_catalog(product_collection, catalog_root)



In [ ]:
# add item metadata

collection = product_id
href = '/home/krasen/Downloads/data_publication/vector_example/eoagritwin-crop-phenology-tomato.parquet/' # to be changed to the online url once we host the data
gdf = gpd.read_parquet(href)
gdf

In [ ]:
minx, miny, maxx, maxy = gdf.total_bounds

min_time, max_time = np.min(gdf.harvest_date.values.ravel()), np.max(gdf.harvest_date.values.ravel())
start = pd.Timestamp(min_time).tz_localize("UTC").to_pydatetime()
end = pd.Timestamp(max_time).tz_localize("UTC").to_pydatetime()

In [ ]:
end = pd.Timestamp(max_time).tz_localize("UTC").to_pydatetime()

item = pystac.Item(
    id="eoagritwin-crop-phenology-tomato-parquet",
    geometry=mapping(box(minx, miny, maxx, maxy)),
    bbox=[minx, miny, maxx, maxy],
    datetime=start,
    properties={'title': 'Tomato Phenology & Crop Classification Vector Layer'},
    
)

proj = ProjectionExtension.ext(item, add_if_missing=True)
proj.code = gdf.crs.to_string()          # e.g. "EPSG:3031"
proj.wkt2 = gdf.crs.to_wkt()
proj.bbox = [minx, miny, maxx, maxy]    # native CRS bbox
proj.geometry = mapping(box(minx, miny, maxx, maxy))

item.extra_fields['title'] = 'Tomato Phenology & Crop Classification Vector Layer'
item

In [ ]:
asset = pystac.Asset(href=href, media_type="application/x-parquet", roles=["data"])

In [ ]:
# add table info to the asset

asset.extra_fields["parquet:columns"] = [
    {
        "Column Name": "parcel_id",
        "Data Type": "int64",
        "Description": "Unique parcel identifier (Varda uid)",
    },
    {
        "Column Name": "campaign_year",
        "Data Type": "int32",
        "Description": "Crop campaign year (2020-2025)",
    },
    {
        "Column Name": "p_tomato",
        "Data Type": "float64",
        "Description": "Calibrated tomato classification probability",
    },
    {
        "Column Name": "is_tomato",
        "Data Type": "int32",
        "Description": "Binary flag (1 if p_tomato >= 0.5, 0 otherwise)",
    },
    {
        "Column Name": "transplant_doy",
        "Data Type": "int32",
        "Description": "Estimated transplanting day of year (1-366)",
    },
    {
        "Column Name": "transplant_date",
        "Data Type": "string",
        "Description": "Estimated transplanting date (YYYY-MM-DD)",
    },
    {
        "Column Name": "harvest_doy",
        "Data Type": "int32",
        "Description": "Estimated harvest day of year (1-366)",
    },
    {
        "Column Name": "harvest_date",
        "Data Type": "string",
        "Description": "Estimated harvest date (YYYY-MM-DD)",
    },
    {
        "Column Name": "los_days",
        "Data Type": "int32",
        "Description": "Length of season in days",
    },
    {
        "Column Name": "geometry",
        "Data Type": "Geometry",
        "Description": "Parcel boundary polygon (EPSG:4326 WGS84)",
    },
]

In [ ]:
item.add_asset('data', asset)

In [ ]:
item.validate()

In [ ]:

catalog_root = Path("../open-science-catalog-metadata").resolve()


In [ ]:
# convert to earthcode item with a reference to a specific open science catalog collection
earthcode_item  = generic_stac_item_to_earthcode_stac_item(item, collection, asset_key='data')

In [ ]:
# save the item to the collection
save_item_to_product_collection(earthcode_item, collection, catalog_root)


In [ ]:
# validate catalog

validate_catalog(catalog_root)

In [ ]:

earthcode_item

